In [1]:
!pip install -q transformers datasets peft torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 87.8 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 68.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 26.1 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 12.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 54.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 15.3 MB/s eta 0:00:00
ERROR: pip's dependency 

In [9]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import get_peft_model, LoraConfig, TaskType
from datasets import Dataset  
from transformers import TrainingArguments, Trainer
import gc

# **Indonesian University Campus Locations for fine-tuning (reference)**

**Universitas Indonesia (UI):**
- Main Campus: Depok, West Java.
- Secondary Campus: Salemba, Central Jakarta (primarily for the Faculty of Medicine and Dentistry).

**Institut Teknologi Bandung (ITB):**
- Main Campus: Jalan Ganesha, Bandung, West Java.
- Secondary Campus: Jatinangor, Sumedang Regency, West Java (houses several newer faculties).

**Universitas Gadjah Mada (UGM):**
- Main Campus: Bulaksumur, Sleman Regency, Special Region of Yogyakarta. The campus is centrally located within the city of Yogyakarta.

**Institut Pertanian Bogor (IPB University):**
- Main Campus: Dramaga, Bogor Regency, West Java.
- Secondary Campus: Baranangsiang, Bogor City (for the business school).

**Universitas Airlangga (Unair):**
- Main Campuses: Spread across Surabaya, East Java, with three main locations: Campus A (Medicine), Campus B (Social Sciences), and Campus C (Science and Technology).

**Institut Teknologi Sepuluh Nopember (ITS):**
- Main Campus: Sukolilo, Surabaya, East Java.
- Secondary Campus: Manyar, Surabaya, East Java (for Civil Engineering).

"""

In [6]:
# Load model dan tokenizer

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print("Loading model... (this may take a while)")
# Load pre-trained model & tokenizer for fine-tuning
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Move to GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

print("Model loaded successfully!")


Loading model... (this may take a while)
Model loaded successfully!


In [7]:
# Structured Q&A from the campus guide for training purposes.

qa_data = [
    {"question": "Where is the main campus of Universitas Indonesia (UI) located?", "answer": "The main campus of Universitas Indonesia is located in Depok, West Java."},
    {"question": "In which city is Institut Teknologi Bandung (ITB) primarily located?", "answer": "ITB's main campus is located in the city of Bandung, West Java."},
    {"question": "Where is the Jatinangor campus of ITB located?", "answer": "The Jatinangor campus of ITB is located in Sumedang Regency, West Java."},
    {"question": "What city is home to Universitas Gadjah Mada (UGM)?", "answer": "Universitas Gadjah Mada is located in Yogyakarta."},
    {"question": "Where is the main campus of Institut Pertanian Bogor (IPB)?", "answer": "The main campus of IPB University is located in Dramaga, Bogor Regency."},
    {"question": "In which city can I find Universitas Airlangga (Unair)?", "answer": "Universitas Airlangga (Unair) is located in Surabaya, East Java, with campuses spread across the city."},
    {"question": "What is the location of the main campus of Institut Teknologi Sepuluh Nopember (ITS)?", "answer": "The main campus of ITS is in the Sukolilo area of Surabaya, East Java."},
    {"question": "Does Universitas Indonesia have a campus in Jakarta?", "answer": "Yes, Universitas Indonesia has a secondary campus in Salemba, Central Jakarta."}
]



In [10]:
# Data preprocessing function
def format_qa(example):
    return {
        "text": f"Question: {example['question']} Answer: {example['answer']}"
    }

qa_dataset = Dataset.from_list(qa_data)
formatted_dataset = qa_dataset.map(format_qa)

# Tokenization function
def preprocess_function(examples):
    inputs = tokenizer(
        examples['text'],
        truncation=True,
        padding="max_length",
        max_length=512
    )
    inputs["labels"] = inputs["input_ids"].copy()
    return inputs

tokenized_qa_dataset = formatted_dataset.map(preprocess_function, batched=True)

# Define LoRA configuration
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
)

# Wrap model with LoRA
model = get_peft_model(model, lora_config)

# Set Training Arguments
training_args = TrainingArguments(
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    warmup_steps=50,
    max_steps=200,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    output_dir="outputs_campus_bot",
    report_to="none",
    remove_unused_columns=False,
)

# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_qa_dataset,
)

# Clear cache and start training
gc.collect()
torch.cuda.empty_cache()
print("GPU cache cleared for training.")
trainer.train()

# Save the fine-tuned model
fine_tuned_model_path = "fine-tuned-Campus-Bot"
model.save_pretrained(fine_tuned_model_path)
tokenizer.save_pretrained(fine_tuned_model_path)
print(f"Fine-tuned model saved to {fine_tuned_model_path}")

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


GPU cache cleared for training.


Step,Training Loss
10,2.136600
20,0.798200
30,0.143300
40,0.062500
50,0.036200
60,0.015000
70,0.006100
80,0.003500
90,0.002900
100,0.002700


Fine-tuned model saved to fine-tuned-Campus-Bot


In [11]:
# !zip -r fine-tuned-Campus-Bot.zip fine-tuned-Campus-Bot

  adding: fine-tuned-Campus-Bot/ (stored 0%)
  adding: fine-tuned-Campus-Bot/tokenizer.json

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


 (deflated 85%)
  adding: fine-tuned-Campus-Bot/tokenizer.model (deflated 55%)
  adding: fine-tuned-Campus-Bot/tokenizer_config.json (deflated 69%)
  adding: fine-tuned-Campus-Bot/README.md (deflated 66%)
  adding: fine-tuned-Campus-Bot/special_tokens_map.json (deflated 79%)
  adding: fine-tuned-Campus-Bot/chat_template.jinja (deflated 60%)
  adding: fine-tuned-Campus-Bot/adapter_config.json (deflated 54%)
  adding: fine-tuned-Campus-Bot/adapter_model.safetensors (deflated 8%)


In [13]:
print(f"\n--- 3. Testing the Fine-Tuned Expert Model from {fine_tuned_model_path} ---")

# Load the fine-tuned model
ft_model = AutoModelForCausalLM.from_pretrained(fine_tuned_model_path)
ft_tokenizer = AutoTokenizer.from_pretrained(fine_tuned_model_path)
ft_model.to(device)

def generate_ft_answer(question, max_length=100):
    prompt = f"Question: {question} Answer:"
    inputs = ft_tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = ft_model.generate(**inputs, max_length=max_length, temperature=0.7, top_k=50, top_p=0.9)
    
    # Clean up the response
    response = ft_tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response.replace(prompt, "").strip()

# Example Test Cases - Known Questions (from training data)
print("\n--- Testing Known Questions ---")
test_questions_known = [
    "Where is the Jatinangor campus of ITB located?",
    "What city is home to Universitas Gadjah Mada (UGM)?",
    "In which city can I find Universitas Airlangga (Unair)?"
]
for q in test_questions_known:
    print(f"Q: {q}")
    print(f"A: {generate_ft_answer(q)}\n")


# Example Test Cases - Unknown Questions (information exists but was not in qa_data)
print("\n--- Testing Unknown Questions (to check generalization) ---")
test_questions_unknown = [
    "Where is the IPB University business school located?",
    "Which faculties are located at the UI Salemba campus?",
    "How many main campuses does Unair have in Surabaya?"
]
for q in test_questions_unknown:
    print(f"Q: {q}")
    print(f"A: {generate_ft_answer(q)}\n")


# Example Test Cases - Wrong/External Question
print("\n--- Testing Wrong/External Question ---")
test_question_wrong = [
    "Where is the National University of Singapore located?"
]
for q in test_question_wrong:
    print(f"Q: {q}")
    print(f"A: {generate_ft_answer(q)}\n")


--- 3. Testing the Fine-Tuned Expert Model from fine-tuned-Campus-Bot ---


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



--- Testing Known Questions ---
Q: Where is the Jatinangor campus of ITB located?


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


A: The Jatinangor campus of ITB is located in Sumedang Regency, West Java.

Q: What city is home to Universitas Gadjah Mada (UGM)?


...

A: The business school of IPB University is located in Depok, West Java.

Q: Which faculties are located at the UI Salemba campus?


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


A: The faculties that are located at the UI Salemba campus are: 1. Business and Economics 2. Humanities and Social Sciences

Q: How many main campuses does Unair have in Surabaya?


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


A: Unair has one main campus in Surabaya.


--- Testing Wrong/External Question ---
Q: Where is the National University of Singapore located?
A: The National University of Singapore (NUS) is located in Singapore.

